# London Clustering, CROCS-Inspired Two-Stage Comparison

Exploratory notebook, first pass, not part of any book chapter. London-side
counterpart to the GoiEner CROCS-inspired notebook, the fourth and final
representation Part 5 Chapter 1 promises get checked against the settled
recipe on real London households.

Every other London clustering notebook collapses each real household to a
single feature vector, a season-mean daily shape or a Tucker/Chronos-2
embedding, then clusters households on that one vector. Yerbury, Campello,
Livingston, Goldsworthy and O'Neil's CROCS (arXiv 2601.10494, already
cited in this book's own `references.bib` as `yerbury2026crocs`) checks a
genuinely different real idea: keep each household's own distinct daily
behaviors separate, and only compare households at the level of "how much
do your typical days actually resemble mine," not a single blended
average. The same real two-stage design as the GoiEner CROCS notebook: a
per-household Representative Load Set (RLS), then a Weighted Sum of
Minimum Distances (WSMD) between households' own RLSs.

Disclosed clearly: this is CROCS-*inspired*, not a reproduction of the
paper's own reported numbers, the same good-faith standard implementation
built for GoiEner, now checked on real London households instead.

## Getting the data

The same shared loader Chapter 2 and Chapter 3 use for London, 1,284
real households after coverage filtering, half-hourly.

In [ ]:
from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.cluster.datasets import load_london_pivot
from ark.cluster.preprocessing import map_to_seasons

LetsPlot.setup_html(isolated_frame=True)
N_PARTITIONS = 50
WINDOW_START = "2013-01-01"
WINDOW_DAYS = 360
MIN_COVERAGE = 0.999
STEPS_PER_DAY = 48  # half-hourly, not hourly like GoiEner

pivot = load_london_pivot(
    n_partitions=N_PARTITIONS,
    window_start=WINDOW_START,
    window_days=WINDOW_DAYS,
    min_coverage=MIN_COVERAGE,
)
n_customers = pivot.shape[1]
household_ids = list(pivot.columns)
print(
    f"pivot: {pivot.shape} (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader"
)

pivot: (17280, 1284) (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader


In [ ]:
def season_tensor(pivot: pd.DataFrame, dates: pd.DatetimeIndex) -> np.ndarray:
    """Real (households, days, half-hours) tensor for a real date subset, households in pivot's own column order."""
    subset = pivot[pivot.index.normalize().isin(dates)]
    return subset.T.to_numpy().reshape(subset.shape[1], len(dates), STEPS_PER_DAY)


day_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
# Real calendar summer (Northern Hemisphere), matching the other London
# and GoiEner notebooks' own convention.
summer_dates = day_index[map_to_seasons(day_index, hemisphere="north") == "summer"]
season = season_tensor(pivot, summer_dates)
print(f"season tensor: {season.shape} (customers, real summer days, half-hours)")

season tensor: (1284, 92, 48) (customers, real summer days, half-hours)


## Stage 1: one Representative Load Set per real household

Each real household's own summer season, normalized by that one
household's own single peak across the whole season, this book's own
magnitude-invariance convention applied once per household. Real days
that behave alike are found by clustering each household's own days
independently, choosing that one household's own $k \in \{2, 3, 4\}$ by
its own real silhouette score.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak


def build_rls(days: np.ndarray, k_range=(2, 3, 4)) -> tuple[np.ndarray, np.ndarray, int]:
    """One household's own Representative Load Set: representative day-shapes plus how often each occurs."""
    best = None
    for k in k_range:
        if len(np.unique(days, axis=0)) <= k:
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(days)
        if len(set(km.labels_)) < 2:
            continue
        score = silhouette_score(days, km.labels_)
        if best is None or score > best[0]:
            best = (score, k, km)
    if best is None:
        # A real, degenerate household whose own days are too uniform to split at all: one real representative.
        return days.mean(axis=0, keepdims=True), np.array([1.0]), 1
    _, k, km = best
    weights = np.array([np.mean(km.labels_ == i) for i in range(k)])
    return km.cluster_centers_, weights, k


rls = [build_rls(season_normalized[h]) for h in range(n_customers)]
household_k = [k for _, _, k in rls]
print(f"real household-level k chosen: min={min(household_k)}, max={max(household_k)}, mean={np.mean(household_k):.2f}")

real household-level k chosen: min=1, max=4, mean=2.53


In [ ]:
from ark.plot.gt_style import themed_gt

household_k_counts = pd.Series(household_k, name="k").value_counts().sort_index().reset_index()
household_k_counts.columns = ["Real per-household k", "Households"]
themed_gt(household_k_counts)

Real per-household k,Households
1,1
2,828
3,233
4,222


## Stage 2: comparing households by their own real RLSs, not one blended average

A Weighted Sum of Minimum Distances, symmetrized across both directions,
the same real construction as the GoiEner CROCS notebook.

In [ ]:
from scipy.spatial.distance import cdist


def wsmd(rls_a, rls_b) -> float:
    centroids_a, weights_a, _ = rls_a
    centroids_b, weights_b, _ = rls_b
    nearest = cdist(centroids_a, centroids_b).min(axis=1)
    return float(np.sum(weights_a * nearest))


n = n_customers
D = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = 0.5 * (wsmd(rls[i], rls[j]) + wsmd(rls[j], rls[i]))
        D[i, j] = D[j, i] = d
print(f"real pairwise distance matrix: {D.shape}, computed from {n} households' own real RLSs")

real pairwise distance matrix: (1284, 1284), computed from 1284 households' own real RLSs


## Clustering households on the real RLS-to-RLS distance

`AgglomerativeClustering` on this real, precomputed distance matrix.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

crocs_rows = []
for k in range(2, 10):
    model = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average")
    labels = model.fit_predict(D)
    score = silhouette_score(D, labels, metric="precomputed")
    crocs_rows.append({"k": k, "silhouette": score})

crocs_scores = pd.DataFrame(crocs_rows)
themed_gt(crocs_scores.round(3))

k,silhouette
2,0.806
3,0.599
4,0.522
5,0.51
6,0.499
7,0.499
8,0.471
9,0.449


In [ ]:
from lets_plot import aes, geom_line, geom_point, ggplot, ggsize, labs

from ark.plot.theme import BRAND_PALETTE, modern_theme

p = (
    ggplot(crocs_scores, aes(x="k", y="silhouette"))
    + geom_line(color=BRAND_PALETTE[0])
    + geom_point(color=BRAND_PALETTE[0], size=4)
    + labs(
        x="Number of household clusters (k)",
        y="Silhouette (precomputed RLS distance)",
        title="Validity by k, CROCS-Inspired RLS Distance, London",
    )
    + modern_theme()
    + ggsize(600, 400)
)
p

In [ ]:
N_CLUSTERS_CROCS = int(crocs_scores.loc[crocs_scores["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_CROCS}")

crocs_model = AgglomerativeClustering(n_clusters=N_CLUSTERS_CROCS, metric="precomputed", linkage="average")
crocs_labels = crocs_model.fit_predict(D)
crocs_table = pd.DataFrame({"labels": crocs_labels}).value_counts().sort_index().reset_index()
crocs_table.columns = ["Label", "Count"]
themed_gt(crocs_table)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,1281
1,3


## Is this split actually trustworthy?

Part 5 Chapter 1's own real trust gate, prediction strength and
cluster-wise stability, both work by fitting Euclidean centroids on part
of the data. CROCS's own WSMD distance has no vector representation for
a centroid to live in, only a precomputed pairwise distance, the same
real incompatibility already disclosed in the GoiEner CROCS notebook.
The same honest substitute applies here: repeated-subsample {{< acr ARI
>}} agreement.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def subsample_agreement_check(
    D: np.ndarray, k: int, n_resamples: int = 15, subsample_frac: float = 0.8, random_state: int = 0
):
    """Repeated-subsample ARI agreement, the honest resampling check for a precomputed-distance clustering."""
    rng = np.random.default_rng(random_state)
    n = D.shape[0]
    subsample_size = int(n * subsample_frac)
    label_sets, index_sets = [], []
    for _ in range(n_resamples):
        idx = rng.choice(n, size=subsample_size, replace=False)
        sub_d = D[np.ix_(idx, idx)]
        model = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average")
        label_sets.append(model.fit_predict(sub_d))
        index_sets.append(idx)

    ari_scores = []
    minority_sizes = []
    for i in range(n_resamples):
        counts = np.bincount(label_sets[i])
        minority_sizes.append(int(counts.min()))
        for j in range(i + 1, n_resamples):
            common = np.intersect1d(index_sets[i], index_sets[j])
            if len(common) < 2:
                continue
            pos_i = {v: p for p, v in enumerate(index_sets[i])}
            pos_j = {v: p for p, v in enumerate(index_sets[j])}
            labels_i = [label_sets[i][pos_i[c]] for c in common]
            labels_j = [label_sets[j][pos_j[c]] for c in common]
            ari_scores.append(adjusted_rand_score(labels_i, labels_j))
    return np.array(ari_scores), np.array(minority_sizes)


ari_scores, minority_sizes = subsample_agreement_check(D, k=N_CLUSTERS_CROCS)
ari_mean, ari_min = ari_scores.mean(), ari_scores.min()
print(f"pairwise ARI across {len(ari_scores)} independent subsample pairs: mean={ari_mean:.3f}, min={ari_min:.3f}")
minority_mean, minority_min, minority_max = minority_sizes.mean(), minority_sizes.min(), minority_sizes.max()
print(f"minority cluster size across 15 subsamples: mean={minority_mean:.1f}, min={minority_min}, max={minority_max}")

pairwise ARI across 105 independent subsample pairs: mean=1.000, min=1.000
minority cluster size across 15 subsamples: mean=2.5, min=1, max=3


## Does accounting for within-household diversity change the real finding?

The settled `shape`+PaCMAP+GMM recipe (Chapter 2) and the no-reduction
shape-only run (Chapter 3) both find London's own $k{=}2$. The real
comparison below is whether letting each household's own day-to-day
diversity speak for itself, rather than averaging it away first, agrees
with that same coarse split or surfaces something different.

## Summary

A real, decisively different shape of result from the GoiEner CROCS
notebook, worth reporting as found rather than forced into the same
mould.

**Most real London households show genuine within-season day-type
diversity.** Only 1 of 1,284 households collapses to a single
representative shape; 828 split cleanly into 2 real day-types, 233 into
3, and 222 into 4, mean $k{=}2.53$. The core CROCS premise, that a
household's own days are not all alike, holds here at least as strongly
as it did on GoiEner.

**The chosen split is extreme, a real 3-household minority against
1,281.** The validity curve falls cleanly from $k{=}2$ (0.806) down
through $k{=}9$ (0.449), a well-behaved, elbow-free shape with no
suspicious jump, unlike the raw-Tucker warning sign found elsewhere in
this comparison. But the real partition itself, at $k{=}2$, isolates only
3 households out of 1,284, a far tighter minority than the settled
`shape`+PaCMAP+GMM recipe's own $k{=}2$ split (982 against 302). Two
representations agree on $k$, not on what that $k$ actually separates.

**Unlike GoiEner, this minority is not a fragile finding, it is about as
stable as a resampling check can show.** Fifteen independent 80%
subsamples of the same 1,284 households, re-clustered from scratch each
time, agree perfectly with each other on the households they share: mean
{{< acr ARI >}} 1.000 across all 105 independent pairs, with the weakest
pair still at 1.000. The minority group itself barely moves either,
ranging only from 1 to 3 households across the fifteen subsamples. This
is a sharp contrast with the GoiEner CROCS notebook's own moderate,
partially reproducible agreement (mean {{< acr ARI >}} 0.527, one pair as
low as $-0.022$). On London, the WSMD-based minority split is not a
borderline signal, it reappears essentially every time this exact
methodology is re-run on a random four-fifths of the population.

Put plainly: CROCS-inspired clustering finds a real, rock-solid structural
signal on London, a tiny, extreme minority of 3 households whose own
day-to-day behaviour genuinely does not resemble the rest of the
population's. That is a real finding in its own right, not a
contradiction of the settled recipe's own coarser 982/302 split; the two
methods are looking at London through different lenses (a season-mean
shape versus a set of per-household representative days) and each lens
surfaces a real, differently-shaped structure. Neither the raw Tucker
notebook's fragile, seed-sensitive splits nor the eventual Chronos-2 and
diffusion-map checks in this same comparison produce agreement this
strong, which makes this the most trustworthy alternative-representation
finding on London checked so far, on its own narrow, extreme-minority
terms.